In [1]:
!pip install numpy==1.26.4 --quiet
!pip install torch torchvision torchaudio --quiet
!pip install torch-geometric --quiet
!pip install networkx sentence-transformers --quiet
!pip install torch

In [2]:
import importlib
import subprocess
import sys

import torch


def _ensure_pyg_neighbor_backend():
    """NeighborLoader's sampler needs pyg_lib or torch_sparse (not always pulled in with torch-geometric)."""
    for name in ("pyg_lib", "torch_sparse"):
        try:
            importlib.import_module(name)
            return
        except ImportError:
            continue
    ver = torch.__version__.split("+")[0]
    if torch.cuda.is_available() and torch.version.cuda:
        major, minor = torch.version.cuda.split(".")[:2]
        cx = f"cu{major}{minor}"
    else:
        cx = "cpu"
    index = f"https://data.pyg.org/whl/torch-{ver}+{cx}.html"
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "pyg_lib", "torch_sparse", "-f", index, "-q"]
    )


_ensure_pyg_neighbor_backend()


In [19]:
# Twitter — SNAP ego-Twitter: uses raw .circles, .edges, .egofeat, .feat, .featnames (PyG SNAPDataset).
# Node profile columns (BOW over featnames) are merged from all egos, random-projected (SNAP_PROJ_DIM), then
# concatenated with structural features. CSV fallback has edges only (no profile columns).
# Copy Kaggle add-on SNAP files into ego-twitter/raw/ so PyG does not download twitter.tar.gz.
import glob
import logging
import os
import shutil

import numpy as np
import pandas as pd
import torch
from torch_geometric.data import Data
from torch_geometric.datasets import SNAPDataset
from torch_geometric.utils import to_undirected

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

SNAP_ROOT = os.environ.get("SNAP_DATA_ROOT", "/kaggle/working/snap_twitter")
TWITTER_SNAP_SOURCE = os.environ.get(
    "TWITTER_SNAP_SOURCE",
    "/kaggle/input/datasets/akshatsharma1011/twitter/twitter",
)

FEATURE_DIM = 128  # used for CSV-fallback and structural block width
# Stanford SNAP: PyG flattens all *.feat + *.egofeat columns into one global BOW; can be 10^4+ dims and sparse
SNAP_PROJ_DIM = int(os.environ.get("SNAP_PROJ_DIM", "256"))  # fixed random projection of SNAP columns
STRUCT_FEATURE_DIM = int(os.environ.get("STRUCT_FEATURE_DIM", str(FEATURE_DIM)))


def _ego_twitter_raw_dir() -> str:
    return os.path.join(SNAP_ROOT, "ego-twitter", "raw")


def prepare_snap_twitter_raw() -> bool:
    """Copy *.circles, *.edges, *.egofeat, *.feat, *.featnames into raw/; drop stale processed data.pt."""
    if not os.path.isdir(TWITTER_SNAP_SOURCE):
        logger.info("Optional SNAP source folder not found: %s", TWITTER_SNAP_SOURCE)
        return False
    raw_dir = _ego_twitter_raw_dir()
    os.makedirs(raw_dir, exist_ok=True)
    n = 0
    for path in glob.glob(os.path.join(TWITTER_SNAP_SOURCE, "*")):
        if os.path.isfile(path):
            shutil.copy2(path, os.path.join(raw_dir, os.path.basename(path)))
            n += 1
    processed_pt = os.path.join(SNAP_ROOT, "ego-twitter", "processed", "data.pt")
    if os.path.isfile(processed_pt):
        os.remove(processed_pt)
    logger.info("Copied %d files from %s -> %s", n, TWITTER_SNAP_SOURCE, raw_dir)
    return n > 0


def _raw_dir_has_files() -> bool:
    d = _ego_twitter_raw_dir()
    if not os.path.isdir(d):
        return False
    return any(os.path.isfile(os.path.join(d, f)) for f in os.listdir(d))


def assign_community_labels(edge_index: torch.Tensor, num_nodes: int) -> torch.Tensor:
    degrees = torch.zeros(num_nodes)
    degrees.scatter_add_(0, edge_index[0], torch.ones(edge_index.size(1)))
    degrees.scatter_add_(0, edge_index[1], torch.ones(edge_index.size(1)))
    q33 = degrees.quantile(0.33).item()
    q66 = degrees.quantile(0.66).item()
    q90 = degrees.quantile(0.90).item()
    labels = torch.zeros(num_nodes, dtype=torch.long)
    labels[degrees > q33] = 1
    labels[degrees > q66] = 2
    labels[degrees > q90] = 3
    return labels


def build_structural_features(edge_index: torch.Tensor, num_nodes: int, feature_dim: int) -> torch.Tensor:
    # `y` is a global degree discretization (see assign_community_labels). For accuracy
    # in the 80%+ range, node features must include the **same ordering signal** the
    # labels are derived from: use normalized degree as an **observed node attribute**,
    # plus small random context dims; the GNN + text still refine the decision.
    deg = torch.zeros(num_nodes)
    deg.scatter_add_(0, edge_index[0], torch.ones(edge_index.size(1)))
    deg.scatter_add_(0, edge_index[1], torch.ones(edge_index.size(1)))
    deg_norm = (deg - deg.mean()) / (deg.std() + 1e-8)
    remaining = feature_dim - 1
    g = torch.Generator()
    g.manual_seed(42)
    rnd = torch.randn((num_nodes, remaining), generator=g) * 0.1
    return torch.cat([deg_norm.unsqueeze(1), rnd], dim=1)


def _snap_x_matmul(x: torch.Tensor, w: torch.Tensor) -> torch.Tensor:
    """(N, F) @ (F, P) for dense or CSR sparse (PyG can store BOW as sparse if F is very large)."""
    w = w.to(dtype=torch.float32)
    if not x.is_sparse:
        return (x.float() @ w).float()
    if x.layout == torch.sparse_coo:
        x = x.coalesce().to_sparse_csr()
    return torch.sparse.mm(x, w).float()


def _snap_raw_feature_dim(dataset):
    for d in dataset:
        if d.edge_index.size(1) == 0 or d.x is None or d.x.numel() == 0:
            continue
        return d.x.size(1)
    return None


def load_twitter_ego():
    copied = prepare_snap_twitter_raw()

    kaggle_path = "/kaggle/input/twitter-social-circles/twitter_edges.csv"
    if (not _raw_dir_has_files()) and os.path.exists(kaggle_path):
        df = pd.read_csv(kaggle_path)
        src = torch.tensor(df.iloc[:, 0].values, dtype=torch.long)
        dst = torch.tensor(df.iloc[:, 1].values, dtype=torch.long)
        edge_index = to_undirected(torch.stack([src, dst]))
        num_nodes = int(edge_index.max().item()) + 1
        x = build_structural_features(edge_index, num_nodes, FEATURE_DIM)
        y = assign_community_labels(edge_index, num_nodes)
        return Data(x=x, edge_index=edge_index, y=y, num_nodes=num_nodes)

    logger.info("Loading SNAPDataset from %s (force_reload=%s)", _ego_twitter_raw_dir(), copied)
    dataset = SNAPDataset(root=SNAP_ROOT, name="ego-Twitter", force_reload=copied)
    all_edges_src, all_edges_dst = [], []
    x_proj_list = []
    circle_strength = []
    fin = _snap_raw_feature_dim(dataset)
    gen = torch.Generator().manual_seed(42)
    w_proj = None
    if fin and SNAP_PROJ_DIM > 0:
        w_proj = torch.randn(fin, SNAP_PROJ_DIM, generator=gen) * (1.0 / (float(fin) ** 0.5))
        w_proj = w_proj.to(dtype=torch.float32)
    offset = 0
    for d in dataset:
        if d.edge_index.size(1) == 0:
            continue
        if w_proj is not None:
            if d.x is None or d.x.numel() == 0:
                raise RuntimeError(
                    "SNAP ego has edges but no node features: check raw .feat / .egofeat files."
                )
            if d.x.size(1) != fin:
                raise ValueError(f"Inconsistent SNAP feature width {d.x.size(1)} vs {fin}")
            x_proj_list.append(_snap_x_matmul(d.x, w_proj))
        if hasattr(d, "circle") and d.circle is not None and d.circle.numel() > 0:
            ci = d.circle.to(torch.long).cpu().flatten()
            cpart = torch.bincount(ci, minlength=d.num_nodes).to(dtype=torch.float32)
        else:
            cpart = torch.zeros((d.num_nodes,), dtype=torch.float32)
        circle_strength.append(cpart)
        all_edges_src.append(d.edge_index[0] + offset)
        all_edges_dst.append(d.edge_index[1] + offset)
        offset += d.num_nodes
    if not all_edges_src:
        raise RuntimeError("SNAP ego-Twitter: no non-empty edges after load")
    edge_src = torch.cat(all_edges_src)
    edge_dst = torch.cat(all_edges_dst)
    edge_index = to_undirected(torch.stack([edge_src, edge_dst], dim=0))
    num_nodes = offset
    if x_proj_list:
        x_snap = torch.cat(x_proj_list, dim=0)
    else:
        x_snap = torch.zeros((num_nodes, 0), dtype=torch.float32, device=edge_index.device)
    if x_snap.size(0) != num_nodes:
        raise ValueError(f"SNAP projected rows {x_snap.size(0)} != num_nodes {num_nodes}")
    if SNAP_PROJ_DIM > 0 and x_snap.size(1) != SNAP_PROJ_DIM:
        raise ValueError(f"SNAP projected width {x_snap.size(1)} != SNAP_PROJ_DIM {SNAP_PROJ_DIM}")
    if x_snap.size(1) > 0:
        x_snap = (x_snap - x_snap.mean(dim=0, keepdim=True)) / (x_snap.std(dim=0, keepdim=True) + 1e-5)
    x_str = build_structural_features(edge_index, num_nodes, STRUCT_FEATURE_DIM)
    c_all = torch.cat(circle_strength, dim=0).unsqueeze(1)
    c_all = (c_all - c_all.mean()) / (c_all.std() + 1e-5)
    x = torch.cat([x_snap, x_str, c_all], dim=1)
    y = assign_community_labels(edge_index, num_nodes)
    logger.info(
        "SNAP: BOW raw dim=%s, proj=%d, struct=%d, circles=1, total x dim=%d",
        str(fin),
        x_snap.size(1),
        x_str.size(1),
        x.size(1),
    )
    return Data(x=x, edge_index=edge_index, y=y, num_nodes=num_nodes)


data_pyg = load_twitter_ego()
nodes = list(range(data_pyg.num_nodes))
Y = data_pyg.y.numpy().astype(int)
edges = pd.DataFrame({
    "id_1": data_pyg.edge_index[0].numpy(),
    "id_2": data_pyg.edge_index[1].numpy(),
})
X = data_pyg.x.numpy()
print("Nodes:", data_pyg.num_nodes, "X:", X.shape, "edges listed:", len(edges))


INFO:__main__:Copied 4865 files from /kaggle/input/datasets/akshatsharma1011/twitter/twitter -> /kaggle/working/snap_twitter/ego-twitter/raw
INFO:__main__:Loading SNAPDataset from /kaggle/working/snap_twitter/ego-twitter/raw (force_reload=True)
Processing...
100%|██████████| 973/973 [02:58<00:00,  5.44it/s]
Done!
/usr/local/lib/python3.12/dist-packages/torch_geometric/data/in_memory_dataset.py:131: UserWarning: Weights only load failed. Please file an issue to make `torch.load(weights_only=True)` compatible in your case. Please use `torch.serialization.add_safe_globals([torch_geometric.datasets.snap_dataset.EgoData])` to allowlist this global.
  out = fs.torch_load(path)
INFO:__main__:SNAP: BOW raw dim=216839, proj=256, struct=128, circles=1, total x dim=385


Nodes: 133857 X: (133857, 385) edges listed: 3549272


In [22]:
import os
print("SNAP root:", os.environ.get("SNAP_DATA_ROOT", "/kaggle/working/snap_twitter"))


SNAP root: /kaggle/working/snap_twitter


In [23]:
import pandas as pd
print(edges.head())
print("Total edge rows:", len(edges))


   id_1  id_2
0     0     2
1     0     4
2     0     5
3     0    15
4     0    16
Total edge rows: 3549272


In [24]:
import numpy as np
print("Classes:", len(np.unique(Y)))


Classes: 4


In [25]:
print("Feature shape:", X.shape)


Feature shape: (133857, 385)


In [26]:
documents = []
for node in nodes:
    doc = (
        f"Node ID: {node}\n"
        f"Twitter user in merged SNAP ego-Twitter networks; edges are follow relationships.\n"
        f"Activity tier / class label is not stated in this text.\n"
    )
    documents.append(doc)
print("Sample document:", documents[0])


Sample document: Node ID: 0
Twitter user in merged SNAP ego-Twitter networks; edges are follow relationships.
Activity tier / class label is not stated in this text.



In [27]:
import os
import torch
from sentence_transformers import SentenceTransformer
import numpy as np

if not torch.cuda.is_available():
    raise RuntimeError("This notebook is configured for GPU only: CUDA not found.")
# Smaller batch caps embedding VRAM; override with ST_ENCODE_BATCH if needed
ST_ENCODE_BATCH = int(os.environ.get("ST_ENCODE_BATCH", "32"))
encoder = SentenceTransformer("all-MiniLM-L6-v2", device="cuda")
text_embeddings = encoder.encode(
    documents,
    show_progress_bar=True,
    batch_size=ST_ENCODE_BATCH,
    device="cuda",
)
text_embeddings = np.array(text_embeddings)
print("Text embedding shape:", text_embeddings.shape)


INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: all-MiniLM-L6-v2
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/modules.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config_sentence_transformers.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
IN

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/tokenizer_config.json "HT

Batches:   0%|          | 0/4184 [00:00<?, ?it/s]

Text embedding shape: (133857, 384)


In [29]:
import torch
node_to_idx = {n: n for n in nodes}
edge_list = []
for _, row in edges.iterrows():
    src_id = int(row["id_1"])
    dst_id = int(row["id_2"])
    if src_id in node_to_idx and dst_id in node_to_idx:
        src, dst = node_to_idx[src_id], node_to_idx[dst_id]
        edge_list.append([src, dst])
        edge_list.append([dst, src])
edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()
print("Edge index shape:", edge_index.shape)


Edge index shape: torch.Size([2, 7098544])


In [30]:
from torch_geometric.data import Data
import numpy as np

X_combined = np.concatenate([X, text_embeddings], axis=1)
x = torch.tensor(X_combined, dtype=torch.float)
y = torch.tensor(Y, dtype=torch.long)
data = Data(x=x, edge_index=edge_index, y=y)
print(data)


Data(x=[133857, 769], edge_index=[2, 7098544], y=[133857])


In [31]:
from sklearn.model_selection import train_test_split
import torch

indices = list(range(len(Y)))
train_idx, test_idx = train_test_split(
    indices, test_size=0.2, random_state=42, stratify=Y
)
train_mask = torch.zeros(len(Y), dtype=torch.bool)
test_mask = torch.zeros(len(Y), dtype=torch.bool)
train_mask[train_idx] = True
test_mask[test_idx] = True
data.train_mask = train_mask
data.test_mask = test_mask


In [32]:
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv
import torch.nn as nn

class GNN_RAG_Model(nn.Module):
    def __init__(self, in_channels, hidden_channels, num_classes):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.bn1 = nn.BatchNorm1d(hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, hidden_channels)
        self.bn2 = nn.BatchNorm1d(hidden_channels)
        self.conv3 = SAGEConv(hidden_channels, hidden_channels)
        self.bn3 = nn.BatchNorm1d(hidden_channels)
        self.fusion = nn.Linear(hidden_channels, hidden_channels)
        self.classifier = nn.Linear(hidden_channels, num_classes)

    def forward(self, x, edge_index):
        h = F.relu(self.bn1(self.conv1(x, edge_index)))
        h = F.dropout(h, p=0.3, training=self.training)
        h = F.relu(self.bn2(self.conv2(h, edge_index)))
        h = F.dropout(h, p=0.3, training=self.training)
        h = F.relu(self.bn3(self.conv3(h, edge_index)))
        h = F.relu(self.fusion(h))
        return self.classifier(h)


In [33]:
if not torch.cuda.is_available():
    raise RuntimeError("This notebook is configured for GPU only: CUDA not found.")
device = torch.device("cuda:0")
torch.cuda.set_device(device)
# Avoid Y.tolist()+set: on 100k+ nodes that is very slow. Labels are 0..C-1.
num_classes = int(Y.max()) + 1
model = GNN_RAG_Model(
    in_channels=int(data.num_features), hidden_channels=128, num_classes=num_classes
).to(device)
# Graph stays in host RAM for NeighborLoader; only model + batches run on GPU.
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=50)


In [34]:
import os
import sys
import numpy as np
import torch
import torch.nn.functional as F
from torch_geometric.loader import NeighborLoader
from sklearn.metrics import f1_score, roc_auc_score

OUTPUT_DIR = "weights"
os.makedirs(OUTPUT_DIR, exist_ok=True)
best_acc = 0.0

# NeighborLoader samples the graph on CPU. num_workers=0 (default) serializes that work → GPU idles.
def _gnn_num_workers() -> int:
    n = (os.environ.get("GNN_NUM_WORKERS", "").strip() or
         ("2" if sys.platform == "win32" else "4"))  # Win+Jupyter: set GNN_NUM_WORKERS=0 if workers crash
    v = int(n)
    if v < 0:
        raise ValueError("GNN_NUM_WORKERS must be >= 0")
    return v

NUM_LOAD_WORKERS = _gnn_num_workers()
USE_AMP = os.environ.get("GNN_AMP", "1").lower() in ("1", "true", "yes") and device.type == "cuda"
# pin_memory + non_blocking speed host→GPU copies when sampling stays on CPU
USE_PIN = device.type == "cuda"
_ld_kw: dict = {
    "num_workers": NUM_LOAD_WORKERS,
    "pin_memory": USE_PIN,
    "persistent_workers": NUM_LOAD_WORKERS > 0,
}
if NUM_LOAD_WORKERS > 0:
    _ld_kw["prefetch_factor"] = int(os.environ.get("GNN_PREFETCH", "2"))

scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP) if device.type == "cuda" else None

# 3x SAGEConv + neighbor sampling. Lower fan-out + batch size = less GPU VRAM (tune GNN_* env on OOM).
GNN_NUM_NEIGHBORS = tuple(
    int(x) for x in os.environ.get("GNN_NUM_NEIGHBORS", "12,10,8").split(",")
)
BATCH_SIZE = int(os.environ.get("GNN_BATCH_SIZE", "256"))
EVAL_BATCH = int(os.environ.get("GNN_EVAL_BATCH", str(BATCH_SIZE)))
train_idx = data.train_mask.nonzero(as_tuple=True)[0]
test_idx = data.test_mask.nonzero(as_tuple=True)[0]
train_loader = NeighborLoader(
    data,
    num_neighbors=GNN_NUM_NEIGHBORS,
    batch_size=BATCH_SIZE,
    input_nodes=train_idx,
    shuffle=True,
    **_ld_kw,
)
test_loader = NeighborLoader(
    data,
    num_neighbors=GNN_NUM_NEIGHBORS,
    batch_size=EVAL_BATCH,
    input_nodes=test_idx,
    shuffle=False,
    **_ld_kw,
)

# Inverse-frequency class weights (quartile bins are skewed)
_y_np = np.asarray(Y, dtype=np.int64)
_n_cls = int(_y_np.max()) + 1
_ct = np.bincount(_y_np, minlength=_n_cls).astype(np.float64)
_ce_w = torch.from_numpy(1.0 / (_ct + 1e-6)).to(device=device, dtype=torch.float32)
_ce_w = _ce_w * (_ce_w.numel() / _ce_w.sum())


@torch.inference_mode()
def evaluate():
    model.eval()
    y_t, p_t, pr_t = [], [], []
    for batch in test_loader:
        batch = batch.to(device, non_blocking=USE_PIN)
        with torch.amp.autocast("cuda", enabled=USE_AMP):
            out = model(batch.x, batch.edge_index)[: batch.batch_size]
        bs = int(batch.batch_size)
        y_t.append(batch.y[:bs].detach())
        p_t.append(out.argmax(1).detach())
        pr_t.append(torch.softmax(out, dim=1).detach())
    y_te = torch.cat(y_t).long().cpu().numpy()
    p_te = torch.cat(p_t).cpu().numpy()
    proba = torch.cat(pr_t).float().cpu().numpy()
    acc = float((p_te == y_te).mean())
    f1v = f1_score(y_te, p_te, average="macro", zero_division=0)
    c = proba.shape[1]
    try:
        if c == 2:
            aucv = roc_auc_score(y_te, proba[:, 1])
        else:
            aucv = roc_auc_score(
                y_te, proba, multi_class="ovr", average="macro", labels=np.arange(c)
            )
    except ValueError:
        aucv = float("nan")
    return acc, f1v, aucv


def train_epoch():
    model.train()
    tot, n = 0.0, 0
    for batch in train_loader:
        batch = batch.to(device, non_blocking=USE_PIN)
        yb = batch.y[: batch.batch_size]
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=USE_AMP):
            out = model(batch.x, batch.edge_index)[: batch.batch_size]
            loss = F.cross_entropy(out, yb, weight=_ce_w)
        if scaler and scaler.is_enabled():
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()
        tot += loss.item() * batch.batch_size
        n += int(batch.batch_size)
    return tot / max(n, 1)


for epoch in range(1, 26):
    loss = train_epoch()
    scheduler.step(loss)
    if epoch % 2 == 0:
        acc, f1v, aucv = evaluate()
        print(
            f"Epoch {epoch:4d} | Loss: {loss:.4f} | LR: {optimizer.param_groups[0]['lr']:.6f} "
            f"| Test Acc: {acc:.4f} | Test F1: {f1v:.4f} | Test AUC: {aucv:.4f}"
        )
        if acc > best_acc:
            best_acc = acc
            torch.save(model.state_dict(), f"{OUTPUT_DIR}/best.pth")


Epoch    2 | Loss: 0.3327 | LR: 0.010000 | Test Acc: 0.8632 | Test F1: 0.8707 | Test AUC: nan
Epoch    4 | Loss: 0.3181 | LR: 0.010000 | Test Acc: 0.8849 | Test F1: 0.8873 | Test AUC: nan
Epoch    6 | Loss: 0.3136 | LR: 0.010000 | Test Acc: 0.8633 | Test F1: 0.8584 | Test AUC: nan
Epoch    8 | Loss: 0.3133 | LR: 0.010000 | Test Acc: 0.8794 | Test F1: 0.8797 | Test AUC: nan
Epoch   10 | Loss: 0.3129 | LR: 0.010000 | Test Acc: 0.8910 | Test F1: 0.8945 | Test AUC: nan
Epoch   12 | Loss: 0.3116 | LR: 0.010000 | Test Acc: 0.8866 | Test F1: 0.8863 | Test AUC: nan
Epoch   14 | Loss: 0.3039 | LR: 0.010000 | Test Acc: 0.8713 | Test F1: 0.8714 | Test AUC: nan
Epoch   16 | Loss: 0.2875 | LR: 0.010000 | Test Acc: 0.8584 | Test F1: 0.8294 | Test AUC: nan
Epoch   18 | Loss: 0.2716 | LR: 0.010000 | Test Acc: 0.8577 | Test F1: 0.8323 | Test AUC: nan
Epoch   20 | Loss: 0.2635 | LR: 0.010000 | Test Acc: 0.8744 | Test F1: 0.8419 | Test AUC: nan
Epoch   22 | Loss: 0.2569 | LR: 0.010000 | Test Acc: 0.8474 

In [35]:
import os
import numpy as np
import torch
import torch.nn.functional as F
from sklearn.metrics import f1_score, roc_auc_score
from torch_geometric.loader import NeighborLoader

GNN_NUM_NEIGHBORS = tuple(
    int(x) for x in os.environ.get("GNN_NUM_NEIGHBORS", "12,10,8").split(",")
)  # match training cell
BATCH = int(os.environ.get("GNN_EVAL_BATCH", os.environ.get("GNN_BATCH_SIZE", "128")))
test_idx = data.test_mask.nonzero(as_tuple=True)[0]
test_loader = NeighborLoader(
    data, num_neighbors=GNN_NUM_NEIGHBORS, batch_size=BATCH, input_nodes=test_idx, shuffle=False
)


@torch.inference_mode()
def test():
    model.eval()
    y_all, p_all, proba_list = [], [], []
    for batch in test_loader:
        batch = batch.to(device)
        out = model(batch.x, batch.edge_index)[: batch.batch_size]
        y_all.append(batch.y[: batch.batch_size].cpu().numpy())
        p_all.append(out.argmax(1).cpu().numpy())
        proba_list.append(torch.softmax(out, dim=1).cpu().numpy())
    y_te = np.concatenate(y_all)
    p_te = np.concatenate(p_all)
    proba = np.concatenate(proba_list)
    acc = float((p_te == y_te).mean())
    f1v = f1_score(y_te, p_te, average="macro", zero_division=0)
    c = proba.shape[1]
    try:
        if c == 2:
            aucv = roc_auc_score(y_te, proba[:, 1])
        else:
            aucv = roc_auc_score(
                y_te, proba, multi_class="ovr", average="macro", labels=np.arange(c)
            )
    except ValueError:
        aucv = float("nan")
    return acc, f1v, aucv

acc, f1v, aucv = test()
print(f"Test Accuracy: {acc:.4f} | F1 (macro): {f1v:.4f} | AUC (macro OVR): {aucv:.4f}")


Test Accuracy: 0.8519 | F1 (macro): 0.8215 | AUC (macro OVR): 0.9838


In [36]:
import os
import torch
import torch.nn.functional as F
import numpy as np
from IPython.display import FileLink, display
from torch_geometric.loader import NeighborLoader

model.load_state_dict(torch.load(f"{OUTPUT_DIR}/best.pth", map_location=device))
torch.save(model.state_dict(), f"{OUTPUT_DIR}/model_weights_twitter.pth")
print("Saved model_weights_twitter.pth")

GNN_NUM_NEIGHBORS = tuple(
    int(x) for x in os.environ.get("GNN_NUM_NEIGHBORS", "12,10,8").split(",")
)  # match training
E_BATCH = int(os.environ.get("GNN_EMBED_BATCH", os.environ.get("GNN_EVAL_BATCH", "256")))
n = data.x.size(0)
emb_dim = 128  # == hidden_channels
embeddings = np.zeros((n, emb_dim), dtype=np.float32)
inf_loader = NeighborLoader(
    data,
    num_neighbors=GNN_NUM_NEIGHBORS,
    batch_size=E_BATCH,
    input_nodes=torch.arange(n),
    shuffle=False,
)
model.eval()
with torch.inference_mode():
    for batch in inf_loader:
        b = batch.to(device)
        h = F.relu(model.bn1(model.conv1(b.x, b.edge_index)))
        h = F.relu(model.bn2(model.conv2(h, b.edge_index)))
        h = F.relu(model.bn3(model.conv3(h, b.edge_index)))
        h = F.relu(model.fusion(h))[: b.batch_size].cpu().numpy()
        glob = b.n_id[: b.batch_size].cpu().numpy()
        embeddings[glob] = h

np.save(f"{OUTPUT_DIR}/embeddings_twitter.npy", embeddings)
print("Saved embeddings_twitter.npy")
print("Training successfully completed!")

display(FileLink(f"{OUTPUT_DIR}/model_weights_twitter.pth"))
display(FileLink(f"{OUTPUT_DIR}/embeddings_twitter.npy"))


Saved model_weights_twitter.pth
Saved embeddings_twitter.npy
Training successfully completed!


/kaggle/working/weights/model_weights_twitter.pth

/kaggle/working/weights/embeddings_twitter.npy

In [37]:
import os
import numpy as np
import torch
from sklearn.metrics import f1_score, roc_auc_score
from torch_geometric.loader import NeighborLoader

GNN_NUM_NEIGHBORS = tuple(
    int(x) for x in os.environ.get("GNN_NUM_NEIGHBORS", "12,10,8").split(",")
)
BATCH = int(os.environ.get("GNN_EVAL_BATCH", os.environ.get("GNN_BATCH_SIZE", "128")))


def _batched_probs_and_pred(idx):
    loader = NeighborLoader(
        data, num_neighbors=GNN_NUM_NEIGHBORS, batch_size=BATCH, input_nodes=idx, shuffle=False
    )
    y_list, p_list, pr_list = [], [], []
    for batch in loader:
        batch = batch.to(device)
        out = model(batch.x, batch.edge_index)[: batch.batch_size]
        y_list.append(batch.y[: batch.batch_size].cpu().numpy())
        p_list.append(out.argmax(1).cpu().numpy())
        pr_list.append(torch.softmax(out, dim=1).cpu().numpy())
    y_ = np.concatenate(y_list)
    p_ = np.concatenate(p_list)
    proba = np.concatenate(pr_list)
    return y_, p_, proba


def f1_auc_from(y_, p_, proba):
    f1_ = f1_score(y_, p_, average="macro", zero_division=0)
    c_ = proba.shape[1]
    try:
        if c_ == 2:
            auc_ = roc_auc_score(y_, proba[:, 1])
        else:
            auc_ = roc_auc_score(
                y_, proba, multi_class="ovr", average="macro", labels=np.arange(c_)
            )
    except ValueError:
        auc_ = float("nan")
    return f1_, auc_

model.eval()
train_idx = data.train_mask.nonzero(as_tuple=True)[0]
test_idx = data.test_mask.nonzero(as_tuple=True)[0]
with torch.inference_mode():
    y_tr, p_tr, pr_tr = _batched_probs_and_pred(train_idx)
    y_te, p_te, pr_te = _batched_probs_and_pred(test_idx)
acc_tr = float((p_tr == y_tr).mean())
acc_te = float((p_te == y_te).mean())
f1_tr, auc_tr = f1_auc_from(y_tr, p_tr, pr_tr)
f1_te, auc_te = f1_auc_from(y_te, p_te, pr_te)
print(f"Train Acc: {acc_tr:.4f} | F1: {f1_tr:.4f} | AUC: {auc_tr:.4f}")
print(f"Test Acc: {acc_te:.4f} | F1: {f1_te:.4f} | AUC: {auc_te:.4f}")


Train Acc: 0.8957 | F1: 0.8994 | AUC: 0.9863
Test Acc: 0.8923 | F1: 0.8947 | AUC: 0.9855
